In [1]:
from ultralytics import YOLO
import cv2
import math
from PIL import Image
from IPython.display import display, clear_output
from collections import deque
import numpy as np

In [6]:
# 1. Load the YOLO model (nano version is fast for real-time)
model = YOLO('yolo26n.pt')  # or 'yolo11n.pt'


# 2. Open the webcam (0 is usually the default camera)
cap = cv2.VideoCapture(0)

# Set resolution (optional)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

while True:
    # 3. Read frame from webcam
    ret, frame = cap.read()
    if not ret:
        break

    # 4. Perform YOLO detection on the frame
    # stream=True optimizes memory usage for video
    results = model(frame, stream=True)

    # 5. Visualize results on the frame
    for r in results:
        annotated_frame = r.plot()

    # 6. Display the frame
    # Convert the numpy array (BGR) to a PIL Image (RGB)
    clear_output()
    img = Image.fromarray(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))
    display(img)

    # 7. Press 'q' to break the loop and exit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 8. Release resources
cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 

In [2]:
# Call the pose model 
model = YOLO("yolo26n-pose.pt")


# Joint connection points
left_shoulder = 5
right_shoulder = 6
left_elbow = 7
right_elbow = 8
left_wrist = 9
right_wrist = 10

# Thresholds
conf_thresh = 0.4
motion_thresh = 5
band_tolerance = 25

# Temporal / Time-related
history_len = 25
cooldown_frames = 20


In [3]:
def keypoints_confident(kp, indices, conf_thresh=conf_thresh):
    for i in indices:
        if kp[i, 2] < conf_thresh:
            return False
    return True


def get_largest_person_index(result):
    if result.boxes is None or len(result.boxes) == 0:
        return None

    boxes = result.boxes.xyxy.cpu().numpy()
    areas = []

    for box in boxes:
        x1, y1, x2, y2 = box
        areas.append((x2 - x1) * (y2 - y1))

    return int(np.argmax(areas))


def get_motion_state(kp, prev_kp=None):
    needed = [
        left_shoulder, right_shoulder,
        left_elbow, right_elbow,
        left_wrist, right_wrist
    ]

    if not keypoints_confident(kp, needed):
        return None

    ls = kp[left_shoulder]
    rs = kp[right_shoulder]
    le = kp[left_elbow]
    re = kp[right_elbow]
    lw = kp[left_wrist]
    rw = kp[right_wrist]

    # define vertical band (elbow ↔ shoulder)
    left_min_y = min(ls[1], le[1]) - band_tolerance
    left_max_y = max(ls[1], le[1]) + band_tolerance

    right_min_y = min(rs[1], re[1]) - band_tolerance
    right_max_y = max(rs[1], re[1]) + band_tolerance

    left_in_band = left_min_y <= lw[1] <= left_max_y
    right_in_band = right_min_y <= rw[1] <= right_max_y

    if not (left_in_band and right_in_band):
        return "out"

    if prev_kp is None:
        return "mid"

    if not keypoints_confident(prev_kp, needed):
        return "mid"

    # motion (y increases downward)
    dl = lw[1] - prev_kp[left_wrist, 1]
    dr = rw[1] - prev_kp[right_wrist, 1]

    both_up = (dl < -motion_thresh) and (dr < -motion_thresh)
    both_down = (dl > motion_thresh) and (dr > motion_thresh)

    if both_up:
        return "up"
    elif both_down:
        return "down"
    else:
        return "mid"


def detect_six_seven(history):
    compact = []

    for state in history:
        if state in ["up", "down"]:
            if len(compact) == 0 or compact[-1] != state:
                compact.append(state)

    if len(compact) < 3:
        return False

    last3 = compact[-3:]

    return (
        last3 == ["up", "down", "up"] or
        last3 == ["down", "up", "down"]
    )

In [4]:
cap = cv2.VideoCapture(0)

history = deque(maxlen=history_len)
prev_kp = None

cooldown = 0
detection_timer = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    annotated = frame.copy()
    current_state = None

    # Run YOLO pose
    results = model(frame, verbose=False)

    if len(results) > 0:
        result = results[0]

        person_idx = get_largest_person_index(result)

        if result.keypoints is not None and person_idx is not None:
            kpts = result.keypoints.data

            if kpts is not None and len(kpts) > person_idx:
                kp = kpts[person_idx].cpu().numpy()

                # Get motion state
                current_state = get_motion_state(kp, prev_kp)

                # Update history
                history.append(current_state)

                # Detect gesture
                detected = False
                if cooldown == 0 and detect_six_seven(history):
                    detected = True
                    detection_timer = 20
                    cooldown = cooldown_frames

                prev_kp = kp

                # Draw YOLO output
                annotated = result.plot()
            else:
                history.append(None)
                prev_kp = None
        else:
            history.append(None)
            prev_kp = None
    else:
        history.append(None)
        prev_kp = None

    # Update timers
    if cooldown > 0:
        cooldown -= 1

    if detection_timer > 0:
        detection_timer -= 1

    # Overlay text
    cv2.putText(
        annotated,
        f"state: {current_state}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    if detection_timer > 0:
        cv2.putText(
            annotated,
            "6 7 DETECTED",
            (20, 90),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.2,
            (0, 0, 255),
            3
        )

    # Show frame
    cv2.imshow("6 7 detector", annotated)

    # Exit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

KeyboardInterrupt: 

In [5]:
cap.release()
cv2.destroyAllWindows()